In [ ]:
from matplotlib import pyplot as plt
%matplotlib inline
from sklearn.preprocessing import LabelEncoder
import keras
import pandas as pd
import numpy as np
from PIL import Image
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
labels = pd.read_csv('cifar10_Labels.csv', index_col=0)
# View an image
img_idx = 5
print(labels.label[img_idx])
Image.open('cifar10/'+str(img_idx)+'.png')

In [ ]:
# Splitting data into Train and Test data
from sklearn.model_selection import train_test_split
y_train, y_test = train_test_split(labels.label, test_size=0.3, random_state=42)
train_idx, test_idx = y_train.index, y_test.index # Stroing indexes for later use
# Reading images for training 
temp = []
for img_idx in y_train.index:
    img_path = os.path.join('cifar10/', str(img_idx) + '.png')
    img = np.array(Image.open(img_path)).astype('float32')        
    temp.append(img)    
X_train = np.stack(temp)
# Reading images for testing 
temp = []
for img_idx in y_test.index:
    img_path = os.path.join('cifar10/', str(img_idx) + '.png')
    img = np.array(Image.open(img_path)).astype('float32')        
    temp.append(img)
X_test = np.stack(temp)
# Normalizing image data
X_train = X_train/255.
X_test = X_test/255.

In [ ]:
# One-hot encoding 10 output classes
encode_X = LabelEncoder()
encode_X_fit = encode_X.fit_transform(y_train)
y_train = keras.utils.np_utils.to_categorical(encode_X_fit)

In [ ]:
# Defining CNN network
num_classes = 10
model = keras.models.Sequential([    
    # Adding first convolutional layer
    keras.layers.Conv2D(filters=32, kernel_size=(3, 3), strides=1, padding='same', activation='relu', 
                        kernel_regularizer=keras.regularizers.l2(0.001), input_shape=(32, 32, 3), name='Conv_1'), 
    # Normalizing the parameters from last layer to speed up the performance (optional)
    keras.layers.BatchNormalization(name='BN_1'),
    # Adding first pooling layer
    keras.layers.MaxPool2D(pool_size=(2, 2), name='MaxPool_1'),
    # Adding second convolutional layer
    keras.layers.Conv2D(filters=64, kernel_size=(3, 3), strides=1, padding='same', activation='relu', 
                        kernel_regularizer=keras.regularizers.l2(0.001), name='Conv_2'),    
    keras.layers.BatchNormalization(name='BN_2'),
    # Adding second pooling layer
    keras.layers.MaxPool2D(pool_size=(2, 2), name='MaxPool_2'),
    # Flattens the input
    keras.layers.Flatten(name='Flat'),
    # Fully-Connected layer
    keras.layers.Dense(num_classes, activation='softmax', name='pred_layer')    
])

In [ ]:
model.summary()

In [ ]:
# Compiling the model
model.compile(loss='categorical_crossentropy', 
              optimizer=keras.optimizers.Adam(), 
              metrics=['accuracy'])
cpfile = r'CIFAR10_checkpoint.hdf5' # Weights to be stored in HDF5 format
cb_checkpoint = keras.callbacks.ModelCheckpoint(cpfile, monitor='val_acc', verbose=1, save_best_only=True, mode='max')
epochs = 5
model.fit(X_train, y_train, epochs=epochs, validation_split=0.2, callbacks=[cb_checkpoint])

In [ ]:
# << DeprecationWarning: The truth value of an empty array is ambiguous >> can arise due to a NumPy version higher than 1.13.3.
# The issue will be updated in upcoming version.
pred = encode_X.inverse_transform(model.predict_classes(X_test[:10])) 
act = y_test[:10]
res = pd.DataFrame([pred, act]).T
res.columns = ['predicted', 'actual']
res

In [ ]:
from mlxtend.evaluate import scoring
train_acc = scoring(encode_X.inverse_transform(model.predict_classes(X_train)),
                   encode_X.inverse_transform([np.argmax(x) for x in y_train]))
test_acc = scoring(encode_X.inverse_transform(model.predict_classes(X_test)), y_test)
print('Train accuracy: ', np.round(train_acc, 5))
print('Test accuracy: ', np.round(test_acc, 5))

In [ ]:
from mlxtend.evaluate import confusion_matrix
from mlxtend.plotting import plot_confusion_matrix
def plot_cm(cm, text):    
    class_names=['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
    plot_confusion_matrix(conf_mat=cm, 
                          colorbar=True, figsize=(8, 8), cmap='Greens',
                          show_absolute=False, show_normed=True)
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45, fontsize=12)
    plt.yticks(tick_marks, class_names, fontsize=12)
    plt.xlabel('Predicted label', fontsize=14)
    plt.ylabel('True label', fontsize=14)
    plt.title(text, fontsize=19, weight='bold')
    plt.show()
# Train Accuracy    
train_cm = confusion_matrix(y_target=encode_X.inverse_transform([np.argmax(x) for x in y_train]), 
                          y_predicted=encode_X.inverse_transform(model.predict_classes(X_train)), 
                          binary=False)
plot_cm(train_cm, 'Confusion Matrix on Train Data')
# Test Accuracy
test_cm = confusion_matrix(y_target=y_test, 
                          y_predicted=encode_X.inverse_transform(model.predict_classes(X_test)), 
                          binary=False)
plot_cm(test_cm, 'Confusion Matrix on Test Data')

In [ ]:
from vis.visualization import visualize_saliency, visualize_cam, overlay
from vis.utils import utils
# Indexes of categories for our model
classes = encode_y.inverse_transform(np.arange(10))
classes
# array(['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog',
#        'horse', 'ship', 'truck'], dtype=object)
# Fetching the ship image
ship_img = utils.load_img('cifar10/'+str(test_idx[6])+'.png') # can use Image.open() also.
plt.imshow(ship_img)
plt.title('Ship image')
plt.show()

In [ ]:
# Predicting the probability for each of the class
ship_prob = model.predict(X_test[6:7]).ravel().copy()
pd.Series(ship_prob, index=classes).plot.barh()
plt.title('Predicted probability of actual ship image by Conv model')
plt.show()

In [ ]:
# Utility to search for layer index by name. 
layer_idx = utils.find_layer_idx(model, 'pred_layer')
# Swap softmax with linear
model.layers[layer_idx].activation = keras.activations.linear
model = utils.apply_modifications(model)

In [ ]:
plt.figure(figsize=(12,6))
for i in range(len(classes)):
    plt.subplot(2, 5, i + 1)
    grads = visualize_saliency(model, layer_idx, filter_indices=i, seed_input=ship_img, backprop_modifier='guided')
    plt.xticks([])
    plt.yticks([])
    plt.xlabel(classes[i])
    plt.title('p=' + str(np.round(ship_prob[i], 4)))
    plt.imshow(grads, cmap='jet')
plt.show()

In [ ]:
plt.figure(figsize=(12,6))
for i in range(len(classes)):
    plt.subplot(2, 5, i + 1)
    cam_grads = visualize_cam(model, layer_idx, filter_indices=i, seed_input=ship_img, backprop_modifier='guided',
                             penultimate_layer_idx=utils.find_layer_idx(model, 'BN_2'))# batch_normalization_14
    plt.xticks([])
    plt.yticks([])
    plt.xlabel(classes[i])
    plt.title('p=' + str(np.round(ship_prob[i], 4)))
    plt.imshow(overlay(cam_grads, ship_img, alpha=0.3))
plt.show()